# CNN-EXP-049 — Colab/Kaggle training launcher

Restartable CUDA workflow for the R16 coarse-to-fine cascade. Run `SETUP`, `INSPECT`, and `SMOKE` first. Network A is trained one OOF fold per session; all folds write to the same persistent run directory.

In [ ]:
from pathlib import Path
import hashlib, os, shutil, subprocess, sys, tarfile

IS_KAGGLE = Path('/kaggle').exists()
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# EDIT THESE PATHS. Kaggle paths refer to attached Datasets; Colab paths refer to Drive.
if IS_KAGGLE:
    CODE_BUNDLE = Path('/kaggle/input/epic-exp049-code/epic-exp049-code.zip')
    DATA_ARCHIVE = Path('/kaggle/input/epic-nematostella/jaNemVect1.1.tgz')
    RISK_BUNDLE = Path('/kaggle/input/epic-exp047-risk-cache/exp047-risk-cache.zip')
    WORK_ROOT = Path('/kaggle/working/epic_exp049')
    PERSIST_ROOT = Path('/kaggle/working/exp049_run')
    # Attach the previous notebook output and edit this path when resuming.
    RESUME_BUNDLE = Path('/kaggle/input/EDIT-PREVIOUS-OUTPUT/exp049_run.zip')
    EXPORT_BUNDLE = Path('/kaggle/working/exp049_run.zip')
else:
    CODE_BUNDLE = Path('/content/drive/MyDrive/EPIC/epic-exp049-code.zip')
    DATA_ARCHIVE = Path('/content/drive/MyDrive/EPIC/jaNemVect1.1.tgz')
    RISK_BUNDLE = Path('/content/drive/MyDrive/EPIC/exp047-risk-cache.zip')
    WORK_ROOT = Path('/content/epic_exp049')
    # Train/read memmaps on the VM SSD; only the compressed resume file lives on Drive.
    PERSIST_ROOT = Path('/content/exp049_run')
    RESUME_BUNDLE = Path('/content/drive/MyDrive/EPIC/exp049_run.zip')
    EXPORT_BUNDLE = RESUME_BUNDLE

FOLDS = 3
FOLD_TO_RUN = 0       # change to 1 and 2 in later sessions
A_STEPS = 20000
B_STEPS = 50000
MAX_MINUTES = 300     # graceful checkpoint before a typical session limit
FILM_BLOCKS = '2,4,6,8'
EXPECTED_MD5 = '733dccb6393a3c07b6e0d7ffaed3686f'
print('platform:', 'kaggle' if IS_KAGGLE else 'colab/local')
print('bundle:', CODE_BUNDLE)
print('data:', DATA_ARCHIVE)
print('EXP047 risk bundle:', RISK_BUNDLE, '(optional)')
print('local run:', PERSIST_ROOT)
print('resume bundle:', RESUME_BUNDLE, '(used only when it exists)')
print('export bundle:', EXPORT_BUNDLE)

## SETUP
Locate and prepare code, Nematostella data, and risk cache. Kaggle may expose either the original archives or their automatically expanded contents; both layouts are supported. This cell does not overwrite the persistent run directory.

In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
def discover_file(configured, filename):
    if configured.is_file():
        return configured
    matches = list(KAGGLE_INPUT.rglob(filename)) if IS_KAGGLE else []
    if len(matches) > 1:
        raise RuntimeError(f'multiple {filename} files: {matches}')
    return matches[0] if matches else configured
CODE_BUNDLE = discover_file(CODE_BUNDLE, 'epic-exp049-code.zip')
DATA_ARCHIVE = discover_file(DATA_ARCHIVE, 'jaNemVect1.1.tgz')
RISK_BUNDLE = discover_file(RISK_BUNDLE, 'exp047-risk-cache.zip')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
if CODE_BUNDLE.is_file():
    shutil.unpack_archive(str(CODE_BUNDLE), str(WORK_ROOT))
else:
    # Kaggle normally expands uploaded ZIP datasets automatically.
    input_runners = list(KAGGLE_INPUT.rglob('scripts/run_cnn_exp049_cloud.py'))
    assert len(input_runners) == 1, ('code bundle/runner not found', input_runners)
    source_project = input_runners[0].parents[1]
    shutil.copytree(source_project, WORK_ROOT / 'project', dirs_exist_ok=True)
if RISK_BUNDLE.is_file():
    shutil.unpack_archive(str(RISK_BUNDLE), str(WORK_ROOT))
runner_matches = list(WORK_ROOT.rglob('scripts/run_cnn_exp049_cloud.py'))
assert len(runner_matches) == 1, runner_matches
PROJECT_ROOT = runner_matches[0].parents[1]
unpacked = PROJECT_ROOT / 'data/raw/unpacked'
assembly_dir = unpacked / 'jaNemVect1.1'
if not assembly_dir.is_dir():
    unpacked.mkdir(parents=True, exist_ok=True)
    if DATA_ARCHIVE.is_file():
        md5 = hashlib.md5()
        with DATA_ARCHIVE.open('rb') as stream:
            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                md5.update(chunk)
        digest = md5.hexdigest()
        assert digest == EXPECTED_MD5, (digest, EXPECTED_MD5)
        with tarfile.open(DATA_ARCHIVE, 'r:gz') as archive:
            archive.extractall(unpacked)
    else:
        # The .tgz may also have been expanded by Kaggle.
        genomes = list(KAGGLE_INPUT.rglob('jaNemVect1.1/genome/genome.fa.gz'))
        assert len(genomes) == 1, ('Nematostella archive/directory not found', genomes)
        shutil.copytree(genomes[0].parents[1], assembly_dir, dirs_exist_ok=True)
assert (assembly_dir / 'genome/genome.fa.gz').is_file(), assembly_dir
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
if RESUME_BUNDLE.is_file() and not any(PERSIST_ROOT.iterdir()):
    print('restoring previous run from:', RESUME_BUNDLE)
    shutil.unpack_archive(str(RESUME_BUNDLE), str(PERSIST_ROOT))
RUNNER = PROJECT_ROOT / 'scripts/run_cnn_exp049_cloud.py'
CACHE_DIR = PERSIST_ROOT / 'regional_cache'
risk_roots = [WORK_ROOT, KAGGLE_INPUT] if IS_KAGGLE else [WORK_ROOT]
risk_cache_matches = list(dict.fromkeys(p for root in risk_roots for p in root.rglob('train_negative_stratum_u8.dat')))
risk_manifest_matches = list(dict.fromkeys(p for root in risk_roots for p in root.rglob('risk_strata_manifest.json')))
RISK_ARGS = ()
if len(risk_cache_matches) == 1 and len(risk_manifest_matches) == 1:
    RISK_ARGS = ('--risk-cache', risk_cache_matches[0], '--risk-manifest', risk_manifest_matches[0])
print('project:', PROJECT_ROOT)
print('runner:', RUNNER)
print('cache:', CACHE_DIR)
print('risk loss:', 'EXP047 exact weights' if RISK_ARGS else 'base BCE pilot')

In [ ]:
def run_stage(stage, *extra):
    command = [
        sys.executable, '-u', str(RUNNER), stage,
        '--run-dir', str(PERSIST_ROOT),
        '--cache-dir', str(CACHE_DIR),
        '--folds', str(FOLDS),
        '--device', 'cuda',
        *map(str, extra),
    ]
    print('RUN:', subprocess.list2cmdline(command), flush=True)
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'stage {stage} failed with exit code {code}')

## INSPECT + SMOKE

In [ ]:
run_stage('inspect')
run_stage('smoke', '--film-blocks', FILM_BLOCKS)

## OOF NETWORK A
Run the following cell for fold 0, then change `FOLD_TO_RUN` to 1 and 2 in later sessions. Resume is automatic. If `MAX_MINUTES` stops training early, `cache-a` will intentionally refuse the incomplete checkpoint; rerun the same cell until the fold reaches `A_STEPS`.

In [ ]:
run_stage(
    'train-a', '--fold', FOLD_TO_RUN, '--a-steps', A_STEPS, '--a-batch', 2,
    '--max-minutes', MAX_MINUTES, '--save-every', 500,
)
run_stage('cache-a', '--fold', FOLD_TO_RUN, '--cache-part', 'train')

## NETWORK B
Run only after all three OOF folds have populated the shared train cache. First, `evaluate-a` measures OOF regional AP and empty-region suppression at 99.5% active recall. It is a diagnostic gate, not the final nucleotide AP. The runner rejects missing, in-sample, and fold-mismatched cache entries.

In [ ]:
run_stage('evaluate-a')
run_stage(
    'train-b', '--film-blocks', FILM_BLOCKS, '--b-steps', B_STEPS, '--b-batch', 2,
    '--max-minutes', MAX_MINUTES, '--save-every', 500, *RISK_ARGS,
)

## FINAL A + VALIDATION CACHE
Train A on every train contig only after the OOF cache and B training are complete.

In [ ]:
run_stage(
    'train-a', '--fold', -1, '--a-steps', A_STEPS, '--a-batch', 2,
    '--max-minutes', MAX_MINUTES, '--save-every', 500,
)
run_stage('cache-a', '--fold', -1, '--cache-part', 'validation')

## ONE NATURAL VALIDATION PASS

In [ ]:
run_stage('evaluate-b', '--film-blocks', FILM_BLOCKS, '--eval-batch', 2)

## EXPORT / RESUME
Run this after every stopped or completed training session. Colab writes one sequential archive to Drive. Kaggle writes it to notebook output: save a version, attach that output on the next session, and set `RESUME_BUNDLE` to its read-only input path.

In [ ]:
EXPORT_BUNDLE.parent.mkdir(parents=True, exist_ok=True)
temporary_base = EXPORT_BUNDLE.with_name(EXPORT_BUNDLE.stem + '.new').with_suffix('')
archive = Path(shutil.make_archive(str(temporary_base), 'zip', PERSIST_ROOT))
os.replace(archive, EXPORT_BUNDLE)
print('saved resume archive:', EXPORT_BUNDLE)